In [ ]:
# Cell 1 — Imports
import os
import warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')
print('✅ Imports OK')

In [ ]:
# Cell 2 — Load & sample data
df = pd.read_pickle('../data/processed/full_dataset.pkl')
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

df_sample = (df.groupby(['ticker', 'year'])
               .first()
               .reset_index())

print(f'Total transcripts:  {len(df)}')
print(f'Sampled:            {len(df_sample)}')
print(f'Unique companies:   {df["ticker"].nunique()}')

In [ ]:
# Cell 3 — Build documents & chunks
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = []
for _, row in df_sample.iterrows():
    text = ' '.join(str(row['transcript']).split()[:1000])
    docs.append(Document(
        page_content=text,
        metadata={
            'ticker':     row['ticker'],
            'date':       str(row['date'].date()),
            'quarter':    row['q'],
            'direction':  int(row['direction']),
            'pct_change': float(row['pct_change'])
        }
    ))

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f'Documents: {len(docs)}')
print(f'Chunks:    {len(chunks)}')

In [ ]:
# Cell 4 — Build FAISS tiny index (only run once)
# Uses 200 companies, 200 words each = 2.8MB index
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os

if not os.path.exists('../data/processed/faiss_tiny'):
    print('Building tiny index...')
    df_tiny = df.groupby('ticker').first().reset_index().head(200)
    tiny_docs = []
    for _, row in df_tiny.iterrows():
        text = ' '.join(str(row['transcript']).split()[:200])
        tiny_docs.append(Document(
            page_content=text,
            metadata={
                'ticker':    row['ticker'],
                'date':      str(row['date'].date()),
                'quarter':   row['q'],
                'direction': int(row['direction']),
                'pct_change': float(row['pct_change'])
            }
        ))
    tiny_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
    tiny_chunks = tiny_splitter.split_documents(tiny_docs)
    print(f'Chunks: {len(tiny_chunks)}')
    
    embeddings = HuggingFaceEmbeddings(
        model_name='sentence-transformers/all-MiniLM-L6-v2',
        model_kwargs={'device': 'cpu'}
    )
    vs = FAISS.from_documents(tiny_chunks, embeddings)
    vs.save_local('../data/processed/faiss_tiny')
    print('✅ Tiny index built and saved!')
else:
    print('✅ Tiny index already exists, skipping build')

In [ ]:
# Cell 5 — Load embeddings + index + Groq client
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

print('Loading embeddings...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'}
)

print('Loading index...')
vs = FAISS.load_local(
    '../data/processed/faiss_tiny',
    embeddings,
    allow_dangerous_deserialization=True
)

client = Groq(api_key=os.getenv('GROQ_API_KEY'))
print('✅ All loaded!')

In [ ]:
# Cell 6 — Define ask function
def ask(question, k=3):
    docs = vs.similarity_search(question, k=k)
    context = '\n'.join([
        f"[{d.metadata['ticker']} | {d.metadata['date']} | {d.metadata['quarter']}]\n{d.page_content}"
        for d in docs
    ])
    r = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role':'user','content':
            f'Using these earnings calls:\n{context}\n\nQuestion: {question}\nAnswer citing ticker and date:'}],
        max_tokens=400
    )
    answer = r.choices[0].message.content
    print(f'\n❓ {question}')
    print(f'💬 {answer}')
    print(f'\n📄 Sources:')
    for d in docs:
        print(f"   → {d.metadata['ticker']} | {d.metadata['date']} | {d.metadata['quarter']}")
    print('-' * 60)
    return answer

print('✅ ask() ready!')

In [ ]:
# Cell 7 — Test the chatbot
ask('What did companies say about supply chain issues in 2021?')
ask('Which companies mentioned COVID impact on revenue?')
ask('What guidance did tech companies give for 2022?')